# Modelos de pronóstico - Hito 2

Comparación de suavizamiento exponencial, promedio móvil y Holt usando los datos de producción procesados en n8n.

## 1. Datos

In [1]:
import numpy as np
import pandas as pd

# Serie diaria validada obtenida del flujo de n8n
datos = pd.read_csv("Calidad_Hito2_serie_diaria_validada.csv", parse_dates=["fecha"])
datos = datos.sort_values(["linea", "fecha"])
print(f"Registros: {len(datos)} | Desde: {datos.fecha.min().date()} | Hasta: {datos.fecha.max().date()}")
print(datos.groupby("linea").size().to_string())

Registros: 252 | Desde: 2026-01-05 | Hasta: 2026-05-04
linea
Linea A    85
Linea B    83
Linea C    84


## 2. Modelos

In [2]:
def suavizamiento(y, alpha=0.2):
    pred = np.full(len(y), np.nan)
    nivel = y[0]
    for i in range(len(y)):
        if i > 0:
            pred[i] = nivel
        nivel = alpha * y[i] + (1 - alpha) * nivel
    return pred, nivel


def promedio_movil(y, ventana=5):
    pred = np.full(len(y), np.nan)
    for i in range(ventana, len(y)):
        pred[i] = np.mean(y[i-ventana:i])
    return pred, np.mean(y[-ventana:])


def holt(y, alpha, beta):
    pred = np.full(len(y), np.nan)
    nivel = y[0]
    tendencia = (y[4] - y[0]) / 4
    for i in range(1, len(y)):
        pred[i] = nivel + tendencia
        anterior = nivel
        nivel = alpha * y[i] + (1 - alpha) * (nivel + tendencia)
        tendencia = beta * (nivel - anterior) + (1 - beta) * tendencia
    return pred, nivel + tendencia


def mae(real, pred):
    return np.mean(np.abs(real - pred))

## 3. Evaluación

Se usan las últimas 20 observaciones de cada línea como prueba. La elección del modelo se hace con datos anteriores. Como alternativa simple se considera repetir la última tasa observada.

In [3]:
filas = []
seleccion = []

for linea, grupo in datos.groupby("linea"):
    y = grupo["tasa_defectos_pct"].to_numpy(dtype=float)
    inicio_prueba = len(y) - 20
    validacion = slice(25, inicio_prueba)
    prueba = slice(inicio_prueba, len(y))

    exp, prox_exp = suavizamiento(y)
    movil, prox_movil = promedio_movil(y)
    simple = np.r_[np.nan, y[:-1]]  # Alternativa simple: último valor

    # Holt: se prueban parámetros usando solo validación
    mejor_error = np.inf
    for alpha in [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.6, 0.8]:
        for beta in [0.01, 0.05, 0.1, 0.2, 0.3, 0.5]:
            pred, siguiente = holt(y, alpha, beta)
            error = mae(y[validacion], pred[validacion])
            if error < mejor_error:
                mejor_error = error
                mejor_holt, prox_holt = pred, siguiente

    modelos = {
        "Exponencial": (exp, prox_exp),
        "Promedio móvil": (movil, prox_movil),
        "Holt": (mejor_holt, prox_holt),
        "Alternativa simple": (simple, y[-1]),
    }
    errores_val = {nombre: mae(y[validacion], p[validacion]) for nombre, (p, _) in modelos.items()}

    elegido = min(["Exponencial", "Promedio móvil"], key=lambda m: errores_val[m])
    if errores_val["Holt"] <= 0.95 * errores_val[elegido]:
        elegido = "Holt"

    for nombre, (pred, _) in modelos.items():
        filas.append({
            "Línea": linea,
            "Modelo": nombre,
            "MAE validación (pp)": errores_val[nombre],
            "MAE prueba (pp)": mae(y[prueba], pred[prueba]),
        })

    seleccion.append({
        "Línea": linea,
        "Modelo elegido": elegido,
        "MAE prueba (pp)": mae(y[prueba], modelos[elegido][0][prueba]),
        "MAE alternativa (pp)": mae(y[prueba], simple[prueba]),
        "Pronóstico siguiente (%)": modelos[elegido][1],
    })

metricas = pd.DataFrame(filas)
resultado = pd.DataFrame(seleccion)

## 4. Resultados

In [4]:
print("MAE de prueba (puntos porcentuales):")
print(metricas.pivot(index="Línea", columns="Modelo", values="MAE prueba (pp)").round(4).to_string())

print("\nModelos seleccionados por validación:")
print(resultado.round(4).to_string(index=False))

MAE de prueba (puntos porcentuales):
Modelo   Alternativa simple  Exponencial    Holt  Promedio móvil
Línea                                                           
Linea A              0.3325       0.2312  0.2264          0.2501
Linea B              0.7307       0.6948  0.7973          0.6327
Linea C              0.2553       0.2181  0.2191          0.2317

Modelos seleccionados por validación:
  Línea Modelo elegido  MAE prueba (pp)  MAE alternativa (pp)  Pronóstico siguiente (%)
Linea A    Exponencial           0.2312                0.3325                    2.1942
Linea B           Holt           0.7973                0.7307                    6.4097
Linea C    Exponencial           0.2181                0.2553                    1.2174


Los modelos seleccionados fueron Exponencial para A y C, y Holt para B. En la prueba final, el promedio móvil tuvo menor error en B. La selección se hizo antes de evaluar ese tramo.